# Keyword Price Outliers With Name Blacklist Filter

`keyword_price_outliers.ipynb` 흐름을 복사하되, 가격 이상치 분석 전에 크롤링 단계에서 제거할 이름 블랙리스트 필터를 먼저 적용합니다.

블랙리스트 필터 기준:
- `DROP_NAME_KEYWORDS`에 제외할 단어를 입력합니다.
- 크롤링된 `name`에 해당 단어가 포함되면 드랍합니다.
- 예: 케이스, 필름, 강화유리, 맥세이프, 충전기 등 본체 가격 통계를 흐리는 액세서리/구매글/홍보 문구

이 노트북은 CSV 저장보다 결과 확인용 DataFrame 출력에 초점을 둡니다.


In [ ]:
from pathlib import Path
import re

import pandas as pd


def resolve_analysis_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "analysis":
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "handoff").exists():
        return cwd.parent

    candidate = cwd / "code/backend/src/main/python/analysis"
    if candidate.exists():
        return candidate

    nested = cwd / "analysis"
    if nested.exists() and (nested / "handoff").exists():
        return nested

    return cwd

ANALYSIS_DIR = resolve_analysis_dir()
PYTHON_DIR = ANALYSIS_DIR.parent
CRAWLING_RESULTS_DIR = PYTHON_DIR / "crawling" / "results"
TARGET_CSV_NAME = "통합조회_전체_no_filter_20260605_1142.csv"

csv_path = CRAWLING_RESULTS_DIR / TARGET_CSV_NAME
if not csv_path.exists():
    raise FileNotFoundError(f"지정한 no-filter 크롤링 CSV를 찾을 수 없습니다: {csv_path}")

csv_path


In [ ]:
# 여기에 크롤링 단계에서 제외하고 싶은 name 포함 키워드를 추가하세요.
DROP_NAME_KEYWORDS = [
    "삽니다",
    "구매합니다",
    "매입",
    "대여",
    "교환",
    "렌탈",
    "매입",
    "대여",
]


def normalize_search_text(value):
    text = str(value or "").lower()
    text = re.sub(r"(?<=[a-z0-9])plus\b", " 플러스", text)
    text = re.sub(r"\bplus\b|[+＋]", " 플러스 ", text)
    text = re.sub(r"\bpro\b", " 프로 ", text)
    text = re.sub(r"\bmax\b", " 맥스 ", text)
    text = re.sub(r"\bultra\b", " 울트라 ", text)
    text = re.sub(r"[^0-9a-z가-힣\s]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_search_text(value):
    return re.sub(r"\s+", "", normalize_search_text(value))


def build_drop_name_keywords(keywords):
    drop_keywords = []
    seen = set()

    for keyword in keywords:
        original = str(keyword or "").strip()
        normalized = normalize_search_text(original)
        compact = compact_search_text(original)
        key = (normalized, compact)

        if not normalized or key in seen:
            continue

        seen.add(key)
        drop_keywords.append(
            {
                "original": original,
                "normalized": normalized,
                "compact": compact,
            }
        )

    return drop_keywords


DROP_NAME_KEYWORD_MATCHERS = build_drop_name_keywords(DROP_NAME_KEYWORDS)
NORMALIZED_DROP_NAME_KEYWORDS = [keyword["normalized"] for keyword in DROP_NAME_KEYWORD_MATCHERS]


def matched_drop_keywords(name):
    raw_name = str(name or "").lower()
    normalized_name = normalize_search_text(name)
    compact_name = compact_search_text(name)

    matched_keywords = []
    for keyword in DROP_NAME_KEYWORD_MATCHERS:
        raw_keyword = keyword["original"].lower()
        normalized_keyword = keyword["normalized"]
        compact_keyword = keyword["compact"]

        if (
            raw_keyword in raw_name
            or normalized_keyword in normalized_name
            or compact_keyword in compact_name
        ):
            matched_keywords.append(keyword["original"])

    return matched_keywords


In [ ]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")

working_df = df.copy()
working_df["price_numeric"] = pd.to_numeric(working_df["price"], errors="coerce")
working_df = working_df.dropna(subset=["keyword", "name", "price_numeric"])
working_df = working_df[working_df["price_numeric"] > 0].copy()
working_df["price_numeric"] = working_df["price_numeric"].astype(int)

working_df["matched_drop_keywords"] = working_df["name"].apply(matched_drop_keywords)
working_df["name_blacklist_drop"] = working_df["matched_drop_keywords"].apply(bool)
working_df["matched_drop_keywords_text"] = working_df["matched_drop_keywords"].apply(lambda values: " | ".join(values))

filtered_df = working_df[~working_df["name_blacklist_drop"]].copy()
dropped_filter_df = working_df[working_df["name_blacklist_drop"]].copy()

filter_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "keyword_count_before_filter": working_df["keyword"].nunique(),
        "keyword_count_after_filter": filtered_df["keyword"].nunique(),
        "drop_keyword_count": len(NORMALIZED_DROP_NAME_KEYWORDS),
    }
])

filter_overview_df


In [ ]:
filter_keyword_summary_df = working_df.groupby("keyword").agg(
    before_count=("keyword", "count"),
    dropped_by_name_blacklist=("name_blacklist_drop", "sum"),
).reset_index()
filter_keyword_summary_df["after_name_blacklist_count"] = (
    filter_keyword_summary_df["before_count"] - filter_keyword_summary_df["dropped_by_name_blacklist"]
)
filter_keyword_summary_df["name_blacklist_drop_rate"] = (
    filter_keyword_summary_df["dropped_by_name_blacklist"] / filter_keyword_summary_df["before_count"]
)
filter_keyword_summary_df = filter_keyword_summary_df.sort_values(
    ["dropped_by_name_blacklist", "before_count"], ascending=[False, False]
)

filter_keyword_summary_df


In [ ]:
# 1차 name 블랙리스트 필터 적용 후 keyword별 저가 데이터 조회
LOW_PRICE_PER_KEYWORD_LIMIT = 30
LOW_PRICE_KEYWORD = None  # 예: "갤럭시", "5090". 전체 keyword 조회는 None

low_price_review_base_df = filtered_df.copy()

if LOW_PRICE_KEYWORD:
    low_price_review_base_df = low_price_review_base_df[
        low_price_review_base_df["keyword"].astype(str).str.contains(LOW_PRICE_KEYWORD, case=False, na=False)
    ]

low_price_review_df = (
    low_price_review_base_df.sort_values(["keyword", "price_numeric", "platform", "pid"])
    .groupby("keyword", group_keys=False)
    .head(LOW_PRICE_PER_KEYWORD_LIMIT)
    .copy()
)
low_price_review_df["low_price_rank_in_keyword"] = (
    low_price_review_df.groupby("keyword").cumcount() + 1
)

low_price_review_df = low_price_review_df[
    [
        "keyword",
        "low_price_rank_in_keyword",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "status",
        "link",
    ]
].sort_values(["keyword", "low_price_rank_in_keyword"])

print(
    f"조회 keyword 수: {low_price_review_df['keyword'].nunique():,}, "
    f"조회 row 수: {len(low_price_review_df):,}"
)

with pd.option_context("display.max_rows", len(low_price_review_df), "display.max_colwidth", 120):
    display(low_price_review_df)


In [ ]:
dropped_filter_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "matched_drop_keywords_text",
    "status",
    "link",
]].sort_values(["keyword", "platform", "price_numeric"]).head(100)


In [ ]:
LOW_PRICE_MEDIAN_RATIO = 0.2

keyword_stats_df = (
    filtered_df.groupby("keyword")["price_numeric"]
    .agg(
        item_count="count",
        min_price="min",
        q1=lambda values: values.quantile(0.25),
        median_price="median",
        q3=lambda values: values.quantile(0.75),
        max_price="max",
        average_price="mean",
    )
    .reset_index()
)

keyword_stats_df["iqr"] = keyword_stats_df["q3"] - keyword_stats_df["q1"]
keyword_stats_df["iqr_lower_bound"] = (keyword_stats_df["q1"] - 1.5 * keyword_stats_df["iqr"]).clip(lower=0)
keyword_stats_df["median_ratio_lower_bound"] = keyword_stats_df["median_price"] * LOW_PRICE_MEDIAN_RATIO
keyword_stats_df["lower_bound"] = keyword_stats_df[["iqr_lower_bound", "median_ratio_lower_bound"]].max(axis=1)
keyword_stats_df["upper_bound"] = keyword_stats_df["q3"] + 1.5 * keyword_stats_df["iqr"]

keyword_stats_df.sort_values(["item_count", "keyword"], ascending=[False, True])


In [ ]:
outlier_df = filtered_df.merge(
    keyword_stats_df[[
        "keyword",
        "q1",
        "median_price",
        "q3",
        "iqr",
        "iqr_lower_bound",
        "median_ratio_lower_bound",
        "lower_bound",
        "upper_bound",
    ]],
    on="keyword",
    how="left",
)

outlier_df["outlier_type"] = ""
outlier_df.loc[outlier_df["price_numeric"] < outlier_df["lower_bound"], "outlier_type"] = "low"
outlier_df.loc[outlier_df["price_numeric"] > outlier_df["upper_bound"], "outlier_type"] = "high"
outlier_df = outlier_df[outlier_df["outlier_type"] != ""].copy()
outlier_df["price_to_median_ratio"] = outlier_df["price_numeric"] / outlier_df["median_price"]

outlier_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "outlier_type",
    "lower_bound",
    "median_price",
    "upper_bound",
    "price_to_median_ratio",
    "status",
    "link",
]].sort_values(["keyword", "outlier_type", "price_numeric"]).head(100)


In [ ]:
# 가격 이상치 name 분석용 조회 셀
# 필요하면 아래 조건만 바꿔서 다시 실행하세요.
OUTLIER_KEYWORD = None  # 예: "갤럭시", "5090". 전체 조회는 None
OUTLIER_TYPE = None  # "low", "high" 또는 전체 조회는 None
NAME_CONTAINS = ""  # name에 포함된 단어로 추가 검색. 전체 조회는 빈 문자열
OUTLIER_LIMIT = 300

price_outlier_review_df = outlier_df.copy()
price_outlier_review_df["price_gap_from_median"] = (
    price_outlier_review_df["price_numeric"] - price_outlier_review_df["median_price"]
)
price_outlier_review_df["outlier_severity"] = price_outlier_review_df.apply(
    lambda row: row["median_price"] / row["price_numeric"]
    if row["outlier_type"] == "low"
    else row["price_to_median_ratio"],
    axis=1,
)

if OUTLIER_KEYWORD:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["keyword"].astype(str).str.contains(OUTLIER_KEYWORD, case=False, na=False)
    ]

if OUTLIER_TYPE:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["outlier_type"].eq(OUTLIER_TYPE)
    ]

if NAME_CONTAINS:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["name"].astype(str).str.contains(NAME_CONTAINS, case=False, na=False)
    ]

price_outlier_review_df = price_outlier_review_df[
    [
        "keyword",
        "outlier_type",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "median_price",
        "price_gap_from_median",
        "price_to_median_ratio",
        "outlier_severity",
        "lower_bound",
        "upper_bound",
        "status",
        "link",
    ]
].sort_values(
    ["keyword", "outlier_type", "outlier_severity"],
    ascending=[True, True, False],
)

print(f"조회된 가격 이상치 수: {len(price_outlier_review_df):,}")
price_outlier_review_df.head(OUTLIER_LIMIT)


In [ ]:
outlier_summary_df = (
    outlier_df.groupby(["keyword", "outlier_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["low", "high"]:
    if column not in outlier_summary_df.columns:
        outlier_summary_df[column] = 0

outlier_summary_df["total_outlier_count"] = outlier_summary_df["low"] + outlier_summary_df["high"]
outlier_summary_df = outlier_summary_df.merge(
    keyword_stats_df[["keyword", "item_count", "min_price", "median_price", "max_price", "lower_bound", "upper_bound"]],
    on="keyword",
    how="left",
)
outlier_summary_df["outlier_rate"] = outlier_summary_df["total_outlier_count"] / outlier_summary_df["item_count"]

outlier_overview_df = outlier_summary_df[[
    "keyword",
    "item_count",
    "low",
    "high",
    "total_outlier_count",
    "outlier_rate",
    "min_price",
    "median_price",
    "max_price",
    "lower_bound",
    "upper_bound",
]].sort_values(["total_outlier_count", "outlier_rate"], ascending=[False, False])

overall_outlier_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "outlier_rows_after_name_blacklist_filter": len(outlier_df),
        "outlier_rate_after_name_blacklist_filter": len(outlier_df) / len(filtered_df) if len(filtered_df) else 0,
        "low_outlier_rows": int((outlier_df["outlier_type"] == "low").sum()) if len(outlier_df) else 0,
        "high_outlier_rows": int((outlier_df["outlier_type"] == "high").sum()) if len(outlier_df) else 0,
        "low_price_median_ratio": LOW_PRICE_MEDIAN_RATIO,
    }
])

display(overall_outlier_overview_df)
outlier_overview_df


In [ ]:
# name 블랙리스트 필터 + 가격 이상치 제거 후 keyword별 가격 요약 DF
outlier_keys = outlier_df[["keyword", "platform", "pid", "price_numeric"]].copy()
outlier_keys["is_outlier"] = True

clean_price_df = filtered_df.merge(
    outlier_keys[["keyword", "platform", "pid", "price_numeric", "is_outlier"]],
    on=["keyword", "platform", "pid", "price_numeric"],
    how="left",
)
clean_price_df["is_outlier"] = clean_price_df["is_outlier"].fillna(False).astype(bool)
clean_price_df = clean_price_df.loc[~clean_price_df["is_outlier"]].copy()

clean_keyword_price_summary_df = (
    clean_price_df.groupby("keyword")["price_numeric"]
    .agg(
        clean_item_count="count",
        clean_min_price="min",
        clean_max_price="max",
        clean_average_price="mean",
        clean_median_price="median",
    )
    .reset_index()
)

filtered_counts_df = filtered_df.groupby("keyword").size().reset_index(name="name_blacklist_filtered_item_count")
outlier_counts_df = outlier_df.groupby("keyword").size().reset_index(name="removed_price_outlier_count")

clean_keyword_price_summary_df = filtered_counts_df.merge(
    outlier_counts_df,
    on="keyword",
    how="left",
).merge(
    clean_keyword_price_summary_df,
    on="keyword",
    how="left",
)
clean_keyword_price_summary_df["removed_price_outlier_count"] = clean_keyword_price_summary_df["removed_price_outlier_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["clean_item_count"] = clean_keyword_price_summary_df["clean_item_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["removed_price_outlier_rate"] = clean_keyword_price_summary_df["removed_price_outlier_count"] / clean_keyword_price_summary_df["name_blacklist_filtered_item_count"]

clean_keyword_price_summary_df = clean_keyword_price_summary_df[[
    "keyword",
    "name_blacklist_filtered_item_count",
    "removed_price_outlier_count",
    "removed_price_outlier_rate",
    "clean_item_count",
    "clean_min_price",
    "clean_max_price",
    "clean_average_price",
    "clean_median_price",
]].sort_values("keyword")

clean_keyword_price_summary_df


## Keyword 기반 상품명 클러스터링 트리

`clean_price_df`를 입력으로 사용해 크롤링 keyword를 1차 루트로 두고, 각 keyword 내부에서 상품명을 기준으로 클러스터를 만듭니다. 결과는 요약용 DataFrame과 트리 형태 dict 둘 다 확인할 수 있습니다.

In [ ]:
from pathlib import Path

from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


# 클러스터링 조정값입니다. 결과가 너무 잘게 쪼개지면 threshold를 올리고, 너무 뭉치면 낮추세요.
CLUSTER_DISTANCE_THRESHOLD = 0.58
CLUSTER_MIN_ITEMS = 2
CLUSTER_TREE_ITEM_LIMIT = 20
CLUSTER_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
CLUSTER_REVIEW_LIMIT = 200

# 상품명 매칭에 방해되는 거래/상태/배송 문구입니다.
CLUSTER_TRADE_NOISE_PATTERN = re.compile(
    r"판매합니다|판매해요|판매중|판매완료|판매|"
    r"팝니다|팔아요|팔아봅니다|팔아여|"
    r"구매합니다|구해요|구합니다|삽니다|매입|"
    r"급처합니다|급처|급매|일괄판매|일괄|"
    r"택포|착불|배송|직거래|거래|네고|에눌|"
    r"미개봉|새상품|신품|중고|정품|풀박스|풀박|단품|"
    r"교환|대여|렌탈"
)


CLUSTER_NUMERIC_TRACKING_CODE_PATTERN = re.compile(
    r"[\(\[\{]\s*\d{3,}\s*[\)\]\}]|"  # 예: (7818), [25773]
    r"^\s*\d{3,}\s+|"  # 예: 749 갤럭시와이드6
    r"\s+\d{3,}\s*$"  # 예: 갤럭시노트8블루 1925
)


def remove_cluster_numeric_tracking_codes(value):
    text = str(value or "")
    text = CLUSTER_NUMERIC_TRACKING_CODE_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def cluster_normalized_name(value):
    text = remove_cluster_numeric_tracking_codes(value)
    text = normalize_search_text(text)
    text = CLUSTER_TRADE_NOISE_PATTERN.sub(" ", text)
    text = re.sub(r"\b\d{1,3}(?:gb|g|tb)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_keyword_name_clusters(source_df):
    cluster_frames = []
    cluster_tree = []

    for keyword, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_df = keyword_df.copy().reset_index(drop=True)
        keyword_df["cluster_name_text"] = keyword_df["name"].apply(cluster_normalized_name)
        keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df["name"].astype(str)

        if len(keyword_df) < CLUSTER_MIN_ITEMS:
            keyword_df["cluster_id"] = 0
        else:
            vectorizer = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(2, 5),
                min_df=1,
                max_features=5000,
            )
            name_vectors = vectorizer.fit_transform(keyword_df["cluster_name_text"])

            try:
                model = AgglomerativeClustering(
                    n_clusters=None,
                    metric="cosine",
                    linkage="average",
                    distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
                )
            except TypeError:
                model = AgglomerativeClustering(
                    n_clusters=None,
                    affinity="cosine",
                    linkage="average",
                    distance_threshold=CLUSTER_DISTANCE_THRESHOLD,
                )

            keyword_df["cluster_id"] = model.fit_predict(name_vectors.toarray())

        keyword_cluster_summaries = []
        keyword_clusters = []
        for cluster_id, cluster_df in keyword_df.groupby("cluster_id", sort=True):
            price_median = cluster_df["price_numeric"].median()
            representative_row = (
                cluster_df.assign(
                    _price_gap=(cluster_df["price_numeric"] - price_median).abs(),
                    _name_length=cluster_df["cluster_name_text"].str.len(),
                )
                .sort_values(["_price_gap", "_name_length", "price_numeric"])
                .iloc[0]
            )
            representative_name = representative_row["cluster_name_text"]
            original_representative_name = representative_row["name"]
            sample_names = list(dict.fromkeys(cluster_df["cluster_name_text"].astype(str).head(5)))
            original_sample_names = list(dict.fromkeys(cluster_df["name"].astype(str).head(5)))

            keyword_cluster_summaries.append(
                {
                    "keyword": keyword,
                    "cluster_id": int(cluster_id),
                    "representative_name": representative_name,
                    "original_representative_name": original_representative_name,
                    "item_count": len(cluster_df),
                    "min_price": int(cluster_df["price_numeric"].min()),
                    "median_price": float(price_median),
                    "average_price": float(cluster_df["price_numeric"].mean()),
                    "max_price": int(cluster_df["price_numeric"].max()),
                    "sample_names": " | ".join(sample_names),
                    "original_sample_names": " | ".join(original_sample_names),
                }
            )
            keyword_clusters.append(
                {
                    "cluster_id": int(cluster_id),
                    "representative_name": representative_name,
                    "original_representative_name": original_representative_name,
                    "item_count": len(cluster_df),
                    "price_summary": {
                        "min": int(cluster_df["price_numeric"].min()),
                        "median": float(price_median),
                        "average": float(cluster_df["price_numeric"].mean()),
                        "max": int(cluster_df["price_numeric"].max()),
                    },
                    "items": cluster_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(CLUSTER_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        keyword_cluster_summary_df = pd.DataFrame(keyword_cluster_summaries).sort_values(
            ["keyword", "item_count", "median_price"], ascending=[True, False, True]
        )
        cluster_frames.append(keyword_cluster_summary_df)
        cluster_tree.append(
            {
                "keyword": keyword,
                "item_count": len(keyword_df),
                "cluster_count": len(keyword_cluster_summary_df),
                "clusters": sorted(keyword_clusters, key=lambda item: item["item_count"], reverse=True),
            }
        )

    summary_df = pd.concat(cluster_frames, ignore_index=True) if cluster_frames else pd.DataFrame()
    return summary_df, cluster_tree


keyword_cluster_summary_df, keyword_cluster_tree = build_keyword_name_clusters(clean_price_df)

cluster_review_df = keyword_cluster_summary_df.copy()
if CLUSTER_REVIEW_KEYWORD:
    cluster_review_df = cluster_review_df[
        cluster_review_df["keyword"].astype(str).str.contains(CLUSTER_REVIEW_KEYWORD, case=False, na=False)
    ]

CLUSTER_REVIEW_CSV_PATH = ANALYSIS_DIR / "review" / "cluster_review_df.csv"
CLUSTER_REVIEW_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
cluster_review_df.to_csv(CLUSTER_REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")

print(
    f"keyword 수: {len(keyword_cluster_tree):,}, "
    f"cluster 수: {len(keyword_cluster_summary_df):,}, "
    f"대상 상품 수: {len(clean_price_df):,}, "
    f"CSV 저장: {CLUSTER_REVIEW_CSV_PATH}"
)
cluster_review_df.head(CLUSTER_REVIEW_LIMIT)

In [ ]:
# 토큰 단계 상품명 클러스터링 최종본입니다.
# 1) 기존 cluster_normalized_name() 규칙으로 거래 문구/관리번호를 제거합니다.
# 2) keyword별 토큰 사전을 만들고, 공백 없이 붙어 있는 상품명도 긴 토큰 우선으로 다시 토큰화합니다.
# 3) c1 -> c2 -> c3 순서로 이전 토큰 경로 안에서 반복 클러스터링합니다.
# 4) c1, c2, ...를 합친 cluster_product_name을 최종 상품명 후보로 사용합니다.
TOKEN_STAGE_DISTANCE_THRESHOLD = 0.25
TOKEN_STAGE_MIN_CLUSTER_ITEMS = 2
TOKEN_STAGE_MAX_COLUMNS = 8
TOKEN_STAGE_TREE_ITEM_LIMIT = 20
TOKEN_STAGE_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
TOKEN_STAGE_REVIEW_LIMIT = 300
TOKEN_STAGE_MIN_VOCAB_TOKEN_LENGTH = 2
TOKEN_STAGE_MIN_VOCAB_TOKEN_COUNT = 2
TOKEN_STAGE_TOKEN_NGRAM_RANGE = (1, 4)


def split_space_tokens(value):
    return [token for token in str(value or "").split() if token]


def build_keyword_token_vocabulary(names):
    token_counts = {}
    for name in names:
        for token in split_space_tokens(name):
            if len(token) < TOKEN_STAGE_MIN_VOCAB_TOKEN_LENGTH:
                continue
            token_counts[token] = token_counts.get(token, 0) + 1

    vocabulary = [
        token
        for token, count in token_counts.items()
        if count >= TOKEN_STAGE_MIN_VOCAB_TOKEN_COUNT
    ]
    return sorted(vocabulary, key=lambda token: (-len(token), -token_counts[token], token))


def split_attached_token(token, token_vocabulary):
    if not token:
        return []

    matches = []
    occupied = [False] * len(token)
    for vocab_token in token_vocabulary:
        if vocab_token == token or len(vocab_token) >= len(token):
            continue

        start_idx = 0
        while True:
            found_idx = token.find(vocab_token, start_idx)
            if found_idx == -1:
                break

            end_idx = found_idx + len(vocab_token)
            if not any(occupied[found_idx:end_idx]):
                matches.append((found_idx, end_idx, vocab_token))
                for idx in range(found_idx, end_idx):
                    occupied[idx] = True
            start_idx = found_idx + 1

    if not matches:
        return [token]

    tokens = []
    cursor = 0
    for start_idx, end_idx, vocab_token in sorted(matches, key=lambda item: item[0]):
        if cursor < start_idx:
            tokens.append(token[cursor:start_idx])
        tokens.append(vocab_token)
        cursor = end_idx
    if cursor < len(token):
        tokens.append(token[cursor:])

    return [item for item in tokens if item]


def tokenize_cluster_name(value, token_vocabulary):
    tokens = []
    for token in split_space_tokens(value):
        tokens.extend(split_attached_token(token, token_vocabulary))
    return tokens


def choose_representative_token(tokens):
    token_counts = pd.Series(list(tokens)).value_counts()
    if token_counts.empty:
        return ""

    max_count = token_counts.iloc[0]
    candidates = token_counts[token_counts == max_count].index.tolist()
    return sorted(candidates, key=lambda token: (-len(token), token))[0]


def build_token_cluster_labels(tokens):
    tokens = pd.Series(tokens).fillna("").astype(str)
    labels = pd.Series("", index=tokens.index, dtype="object")
    non_empty_tokens = tokens[tokens != ""]
    unique_tokens = non_empty_tokens.drop_duplicates().tolist()

    if not unique_tokens:
        return labels
    if len(unique_tokens) < TOKEN_STAGE_MIN_CLUSTER_ITEMS:
        labels.loc[non_empty_tokens.index] = non_empty_tokens
        return labels

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=TOKEN_STAGE_TOKEN_NGRAM_RANGE,
        min_df=1,
    )
    token_vectors = vectorizer.fit_transform(unique_tokens)

    try:
        model = AgglomerativeClustering(
            n_clusters=None,
            metric="cosine",
            linkage="average",
            distance_threshold=TOKEN_STAGE_DISTANCE_THRESHOLD,
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=None,
            affinity="cosine",
            linkage="average",
            distance_threshold=TOKEN_STAGE_DISTANCE_THRESHOLD,
        )

    token_cluster_df = pd.DataFrame(
        {
            "token": unique_tokens,
            "token_cluster_id": model.fit_predict(token_vectors.toarray()),
        }
    )
    token_frequency = non_empty_tokens.value_counts().to_dict()

    token_label_map = {}
    for _, cluster_df in token_cluster_df.groupby("token_cluster_id", sort=True):
        representative_token = choose_representative_token(
            token
            for token in cluster_df["token"]
            for _ in range(token_frequency.get(token, 0))
        )
        for token in cluster_df["token"]:
            token_label_map[token] = representative_token

    labels.loc[non_empty_tokens.index] = non_empty_tokens.map(token_label_map)
    return labels


def join_cluster_product_name(row, token_columns):
    tokens = [str(row[column]).strip() for column in token_columns if str(row[column]).strip()]
    return " ".join(tokens)


def add_token_stage_columns(keyword_df):
    keyword_df = keyword_df.copy().reset_index(drop=True)
    keyword_df["cluster_name_text"] = keyword_df["name"].apply(cluster_normalized_name)
    keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df[
        "name"
    ].astype(str)

    token_vocabulary = build_keyword_token_vocabulary(keyword_df["cluster_name_text"])
    keyword_df["_name_tokens"] = keyword_df["cluster_name_text"].apply(
        lambda value: tokenize_cluster_name(value, token_vocabulary)
    )
    keyword_df["tokenized_cluster_name_text"] = keyword_df["_name_tokens"].apply(" ".join)

    token_columns = []
    max_token_count = int(keyword_df["_name_tokens"].apply(len).max()) if len(keyword_df) else 0
    for token_position in range(min(max_token_count, TOKEN_STAGE_MAX_COLUMNS)):
        token_column = f"c{token_position + 1}"
        token_columns.append(token_column)
        keyword_df[token_column] = ""

        group_columns = token_columns[:-1] or ["keyword"]
        for _, stage_df in keyword_df.groupby(group_columns, sort=True, dropna=False):
            stage_tokens = stage_df["_name_tokens"].apply(
                lambda tokens: tokens[token_position] if token_position < len(tokens) else ""
            )
            keyword_df.loc[stage_df.index, token_column] = build_token_cluster_labels(stage_tokens).values

    keyword_df["cluster_product_name"] = keyword_df.apply(
        join_cluster_product_name,
        axis=1,
        token_columns=token_columns,
    )
    return keyword_df.drop(columns=["_name_tokens"]), token_columns


def summarize_token_stage_clusters(item_df, token_columns):
    if item_df.empty:
        return pd.DataFrame()

    summary_rows = []
    group_columns = ["keyword", *token_columns, "cluster_product_name"]
    for group_key, cluster_df in item_df.groupby(group_columns, sort=True, dropna=False):
        group_values = dict(zip(group_columns, group_key if isinstance(group_key, tuple) else (group_key,)))
        price_median = cluster_df["price_numeric"].median()
        summary_rows.append(
            {
                **group_values,
                "item_count": len(cluster_df),
                "min_price": int(cluster_df["price_numeric"].min()),
                "median_price": float(price_median),
                "average_price": float(cluster_df["price_numeric"].mean()),
                "max_price": int(cluster_df["price_numeric"].max()),
                "sample_tokenized_names": " | ".join(
                    dict.fromkeys(cluster_df["tokenized_cluster_name_text"].astype(str).head(5))
                ),
                "sample_normalized_names": " | ".join(
                    dict.fromkeys(cluster_df["cluster_name_text"].astype(str).head(5))
                ),
                "sample_original_names": " | ".join(
                    dict.fromkeys(cluster_df["name"].astype(str).head(5))
                ),
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["keyword", "item_count", "median_price"], ascending=[True, False, True]
    )

    front_columns = ["keyword", *token_columns, "cluster_product_name"]
    return summary_df[front_columns + [column for column in summary_df.columns if column not in front_columns]]


def build_token_stage_cluster_tree(item_df, summary_df, token_columns):
    tree_rows = []
    if item_df.empty or summary_df.empty:
        return tree_rows

    for keyword, keyword_summary_df in summary_df.groupby("keyword", sort=True):
        keyword_item_df = item_df[item_df["keyword"] == keyword]
        clusters = []
        for _, summary_row in keyword_summary_df.iterrows():
            mask = keyword_item_df["cluster_product_name"] == summary_row["cluster_product_name"]
            for token_column in token_columns:
                mask &= keyword_item_df[token_column] == summary_row[token_column]
            cluster_item_df = keyword_item_df[mask]
            clusters.append(
                {
                    "cluster_product_name": summary_row["cluster_product_name"],
                    "token_path": {column: summary_row[column] for column in token_columns if summary_row[column]},
                    "item_count": int(summary_row["item_count"]),
                    "price_summary": {
                        "min": int(summary_row["min_price"]),
                        "median": float(summary_row["median_price"]),
                        "average": float(summary_row["average_price"]),
                        "max": int(summary_row["max_price"]),
                    },
                    "items": cluster_item_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(TOKEN_STAGE_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        tree_rows.append(
            {
                "keyword": keyword,
                "item_count": len(keyword_item_df),
                "cluster_count": len(keyword_summary_df),
                "clusters": sorted(clusters, key=lambda item: item["item_count"], reverse=True),
            }
        )

    return tree_rows


def build_keyword_token_stage_clusters(source_df):
    item_frames = []
    token_column_count = 0

    for _, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_item_df, keyword_token_columns = add_token_stage_columns(keyword_df)
        item_frames.append(keyword_item_df)
        token_column_count = max(token_column_count, len(keyword_token_columns))

    item_df = pd.concat(item_frames, ignore_index=True) if item_frames else pd.DataFrame()
    token_columns = [f"c{idx}" for idx in range(1, token_column_count + 1)]
    for token_column in token_columns:
        if token_column not in item_df.columns:
            item_df[token_column] = ""
    item_df[token_columns] = item_df[token_columns].fillna("")

    summary_df = summarize_token_stage_clusters(item_df, token_columns)
    tree_rows = build_token_stage_cluster_tree(item_df, summary_df, token_columns)
    return item_df, summary_df, tree_rows


token_stage_cluster_item_df, token_stage_cluster_summary_df, token_stage_cluster_tree = (
    build_keyword_token_stage_clusters(clean_price_df)
)

token_stage_review_df = token_stage_cluster_summary_df.copy()
if TOKEN_STAGE_REVIEW_KEYWORD:
    token_stage_review_df = token_stage_review_df[
        token_stage_review_df["keyword"].astype(str).str.contains(
            TOKEN_STAGE_REVIEW_KEYWORD, case=False, na=False
        )
    ]

TOKEN_STAGE_REVIEW_CSV_PATH = ANALYSIS_DIR / "review" / "token_stage_cluster_review_df.csv"
TOKEN_STAGE_REVIEW_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
token_stage_review_df.to_csv(TOKEN_STAGE_REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")

print(
    f"keyword 수: {len(token_stage_cluster_tree):,}, "
    f"token-stage cluster 수: {len(token_stage_cluster_summary_df):,}, "
    f"대상 상품 수: {len(token_stage_cluster_item_df):,}, "
    f"CSV 저장: {TOKEN_STAGE_REVIEW_CSV_PATH}"
)
token_stage_review_df.head(TOKEN_STAGE_REVIEW_LIMIT)

## 클러스터링 전 특수문자 리스크 샘플

상품명 정규화 과정에서 의미가 사라질 수 있는 특수문자/기호 패턴을 별도로 확인합니다. 이 결과를 보고 `cluster_normalized_name()`에서 보존하거나 치환할 규칙을 정합니다.

## 최소단계 클러스터링 리뷰

큰 군집으로 묶기 전에, 같은 keyword 안에서 상품명이 거의 같은 항목끼리만 먼저 묶이는 최소단계 결과를 확인합니다. 유사도 기준을 높게 잡아 중복/유사 매물 수준의 작은 묶음을 보는 용도입니다.

In [ ]:
# 클러스터링 전 특수문자/기호가 의미를 가질 수 있는 케이스를 별도로 확인합니다.
SPECIAL_CHAR_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
SPECIAL_CHAR_SAMPLE_LIMIT = 8
SPECIAL_CHAR_REVIEW_LIMIT = 300

SPECIAL_CHAR_PATTERNS = [
    {
        "pattern_id": "plus_symbol_or_word",
        "description": "+/＋/plus: 플러스 모델 구분 가능성",
        "regex": r"(?i)(\+|＋|\bplus\b)",
    },
    {
        "pattern_id": "slash_multiple_models",
        "description": "/: 여러 모델 동시 기재 또는 호환 기종 구분 가능성",
        "regex": r"/",
    },
    {
        "pattern_id": "hyphen_model_token",
        "description": "-: 모델명/세대/등급 연결 가능성",
        "regex": r"(?i)[0-9a-z가-힣]\s*[-–—]\s*[0-9a-z가-힣]",
    },
    {
        "pattern_id": "parentheses_detail",
        "description": "괄호: 상태/구성/세부 모델 정보 포함 가능성",
        "regex": r"[\(\)\[\]\{\}]",
    },
    {
        "pattern_id": "inch_symbol",
        "description": "따옴표/인치: 화면 크기 정보 가능성",
        "regex": r"\d+(?:\.\d+)?\s*(?:\"|”|″|인치)",
    },
    {
        "pattern_id": "storage_unit",
        "description": "GB/TB: 용량 구분 가능성",
        "regex": r"(?i)\d+\s*(?:gb|g|tb)",
    },
    {
        "pattern_id": "roman_generation",
        "description": "I/II/III/IV: 세대 표기 가능성",
        "regex": r"(?i)\b(?:i|ii|iii|iv|v)\b",
    },
    {
        "pattern_id": "ampersand_bundle",
        "description": "&: 번들/동시 기재 가능성",
        "regex": r"&",
    },
    {
        "pattern_id": "remaining_special_chars",
        "description": "기타 특수문자: 정규화 시 공백화되는 잔여 기호",
        "regex": r"[^0-9a-zA-Z가-힣\s]",
    },
]


def matched_special_char_patterns(name):
    text = str(name or "")
    matched = []
    for item in SPECIAL_CHAR_PATTERNS:
        if re.search(item["regex"], text):
            matched.append(item["pattern_id"])
    return matched


special_char_review_base_df = clean_price_df.copy()
if SPECIAL_CHAR_REVIEW_KEYWORD:
    special_char_review_base_df = special_char_review_base_df[
        special_char_review_base_df["keyword"].astype(str).str.contains(
            SPECIAL_CHAR_REVIEW_KEYWORD, case=False, na=False
        )
    ].copy()

special_char_review_df = special_char_review_base_df[
    ["keyword", "platform", "pid", "name", "price_numeric", "status", "link"]
].copy()
special_char_review_df["matched_special_patterns"] = special_char_review_df["name"].apply(
    matched_special_char_patterns
)
special_char_review_df = special_char_review_df[
    special_char_review_df["matched_special_patterns"].apply(bool)
].copy()
special_char_review_df["matched_special_patterns_text"] = special_char_review_df[
    "matched_special_patterns"
].apply(lambda values: " | ".join(values))
special_char_review_df["normalized_for_cluster"] = special_char_review_df["name"].apply(
    cluster_normalized_name if "cluster_normalized_name" in globals() else normalize_search_text
)

special_char_pattern_summary_rows = []
for item in SPECIAL_CHAR_PATTERNS:
    pattern_df = special_char_review_df[
        special_char_review_df["matched_special_patterns"].apply(lambda values: item["pattern_id"] in values)
    ]
    special_char_pattern_summary_rows.append(
        {
            "pattern_id": item["pattern_id"],
            "description": item["description"],
            "matched_row_count": len(pattern_df),
            "keyword_count": pattern_df["keyword"].nunique(),
            "sample_names": " | ".join(
                dict.fromkeys(pattern_df["name"].astype(str).head(SPECIAL_CHAR_SAMPLE_LIMIT))
            ),
        }
    )

special_char_pattern_summary_df = pd.DataFrame(special_char_pattern_summary_rows).sort_values(
    ["matched_row_count", "pattern_id"], ascending=[False, True]
)

print(
    f"특수문자/기호 패턴 포함 상품 수: {len(special_char_review_df):,} / "
    f"검토 대상 상품 수: {len(special_char_review_base_df):,}"
)
display(special_char_pattern_summary_df)

special_char_review_df[
    [
        "keyword",
        "matched_special_patterns_text",
        "name",
        "normalized_for_cluster",
        "price_numeric",
        "platform",
        "status",
        "link",
    ]
].sort_values(["keyword", "matched_special_patterns_text", "price_numeric"]).head(SPECIAL_CHAR_REVIEW_LIMIT)

In [ ]:
from sklearn.neighbors import NearestNeighbors


# 최소단계 묶음 기준입니다. 높일수록 거의 같은 이름만 묶이고, 낮출수록 더 많이 묶입니다.
MIN_STAGE_SIMILARITY_THRESHOLD = 0.92
MIN_STAGE_MIN_ITEMS = 2
MIN_STAGE_TREE_ITEM_LIMIT = 12
MIN_STAGE_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
MIN_STAGE_REVIEW_LIMIT = 300


def find_connected_components(size, edges):
    parent = list(range(size))

    def find(value):
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value

    def union(left, right):
        left_root = find(left)
        right_root = find(right)
        if left_root != right_root:
            parent[right_root] = left_root

    for left, right in edges:
        union(left, right)

    components = {}
    for idx in range(size):
        components.setdefault(find(idx), []).append(idx)
    return list(components.values())


def build_min_stage_name_clusters(source_df):
    summary_rows = []
    tree_rows = []

    for keyword, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_df = keyword_df.copy().reset_index(drop=True)
        keyword_df["cluster_name_text"] = keyword_df["name"].apply(
            cluster_normalized_name if "cluster_normalized_name" in globals() else normalize_search_text
        )
        keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df[
            "name"
        ].astype(str)

        if len(keyword_df) < MIN_STAGE_MIN_ITEMS:
            continue

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=1,
            max_features=5000,
        )
        name_vectors = vectorizer.fit_transform(keyword_df["cluster_name_text"])
        radius = 1 - MIN_STAGE_SIMILARITY_THRESHOLD

        neighbor_model = NearestNeighbors(metric="cosine", algorithm="brute", radius=radius)
        neighbor_model.fit(name_vectors)
        distances_list, indices_list = neighbor_model.radius_neighbors(
            name_vectors,
            return_distance=True,
            sort_results=True,
        )

        edges = []
        edge_similarity_by_pair = {}
        for source_idx, (distances, indices) in enumerate(zip(distances_list, indices_list)):
            for distance, target_idx in zip(distances, indices):
                target_idx = int(target_idx)
                if source_idx >= target_idx:
                    continue
                similarity = 1 - float(distance)
                if similarity >= MIN_STAGE_SIMILARITY_THRESHOLD:
                    edges.append((source_idx, target_idx))
                    edge_similarity_by_pair[(source_idx, target_idx)] = similarity

        components = [
            component
            for component in find_connected_components(len(keyword_df), edges)
            if len(component) >= MIN_STAGE_MIN_ITEMS
        ]

        keyword_clusters = []
        for min_stage_cluster_id, component in enumerate(components):
            component_df = keyword_df.iloc[component].copy()
            price_median = component_df["price_numeric"].median()
            representative_row = (
                component_df.assign(
                    _price_gap=(component_df["price_numeric"] - price_median).abs(),
                    _name_length=component_df["cluster_name_text"].str.len(),
                )
                .sort_values(["_price_gap", "_name_length", "price_numeric"])
                .iloc[0]
            )

            component_set = set(component)
            component_similarities = [
                similarity
                for (left, right), similarity in edge_similarity_by_pair.items()
                if left in component_set and right in component_set
            ]
            min_similarity = min(component_similarities) if component_similarities else 1.0
            max_similarity = max(component_similarities) if component_similarities else 1.0
            average_similarity = (
                sum(component_similarities) / len(component_similarities)
                if component_similarities
                else 1.0
            )

            sample_names = list(dict.fromkeys(component_df["name"].astype(str).head(6)))
            normalized_samples = list(
                dict.fromkeys(component_df["cluster_name_text"].astype(str).head(6))
            )
            summary_row = {
                "keyword": keyword,
                "min_stage_cluster_id": int(min_stage_cluster_id),
                "representative_name": representative_row["name"],
                "item_count": len(component_df),
                "min_pair_similarity": round(min_similarity, 4),
                "avg_pair_similarity": round(average_similarity, 4),
                "max_pair_similarity": round(max_similarity, 4),
                "min_price": int(component_df["price_numeric"].min()),
                "median_price": float(price_median),
                "max_price": int(component_df["price_numeric"].max()),
                "sample_names": " | ".join(sample_names),
                "normalized_samples": " | ".join(normalized_samples),
            }
            summary_rows.append(summary_row)
            keyword_clusters.append(
                {
                    **summary_row,
                    "items": component_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(MIN_STAGE_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        if keyword_clusters:
            tree_rows.append(
                {
                    "keyword": keyword,
                    "item_count": len(keyword_df),
                    "min_stage_cluster_count": len(keyword_clusters),
                    "clustered_item_count": sum(item["item_count"] for item in keyword_clusters),
                    "clusters": sorted(
                        keyword_clusters,
                        key=lambda item: (item["item_count"], item["avg_pair_similarity"]),
                        reverse=True,
                    ),
                }
            )

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            ["keyword", "item_count", "avg_pair_similarity"],
            ascending=[True, False, False],
        ).reset_index(drop=True)
    return summary_df, tree_rows


min_stage_cluster_summary_df, min_stage_cluster_tree = build_min_stage_name_clusters(clean_price_df)

min_stage_review_df = min_stage_cluster_summary_df.copy()
if MIN_STAGE_REVIEW_KEYWORD and not min_stage_review_df.empty:
    min_stage_review_df = min_stage_review_df[
        min_stage_review_df["keyword"].astype(str).str.contains(
            MIN_STAGE_REVIEW_KEYWORD, case=False, na=False
        )
    ]

print(
    f"최소단계 cluster 수: {len(min_stage_cluster_summary_df):,}, "
    f"묶인 상품 수: {min_stage_cluster_summary_df['item_count'].sum() if not min_stage_cluster_summary_df.empty else 0:,}, "
    f"대상 상품 수: {len(clean_price_df):,}, "
    f"유사도 기준: {MIN_STAGE_SIMILARITY_THRESHOLD}"
)
min_stage_review_df.head(MIN_STAGE_REVIEW_LIMIT)